# Chinese Sentiment Classification — IPFA Research Foundation

**Author:** AlreadyJ  
**Purpose:** Text-channel baseline for the *Illocutionary Pragmatic Force Attenuation* (IPFA) research pipeline  
**Model:** `bert-base-chinese` fine-tuned on ChnSentiCorp  
**Runtime:** CPU-compatible (≈ 25–40 min on Colab free tier)

---

## Research context

IPFA describes a failure mode in AI mental health support systems: when a patient's affective output is suppressed (a cardinal symptom of depression and schizophrenia), standard classifiers systematically misread suppressed distress as neutral or positive sentiment — a clinically dangerous false negative.

This notebook trains a text-only Chinese sentiment classifier as a **controlled baseline**. Its purpose is twofold:

1. Establish how well surface-level sentiment labels can be inferred from Chinese text alone
2. Demonstrate the baseline's vulnerability to IPFA — i.e., face-preserving, affectively attenuated language that conceals distress beneath pragmatically neutral surface forms

The IPFA-vulnerable examples in §4 are the conceptual core: they show exactly where vanilla fine-tuning breaks, motivating domain-adversarial adaptation in the full pipeline.

## §1 — Setup

In [ ]:
# Install dependencies (Colab)
!pip install transformers datasets torch scikit-learn pandas seaborn matplotlib -q

In [ ]:
import json, os, time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## §2 — Data loading and tokenisation

In [ ]:
# Hyperparameters — reduce EPOCHS to 1 for a quick smoke-test
MODEL_NAME  = 'bert-base-chinese'
DATASET_NAME = 'dirtycomputer/weibo_senti_100k'
MAX_LEN     = 128
BATCH_SIZE  = 16
EPOCHS      = 3
LR          = 2e-5

In [ ]:
from datasets import load_dataset, concatenate_datasets

ds = load_dataset("dirtycomputer/weibo_senti_100k")
print(ds)
pd.DataFrame(ds['train'].to_pandas()).head(3)

# Split into train/test manually
ds_split = ds['train'].train_test_split(test_size=0.1, seed=42)
ds = {'train': ds_split['train'], 'test': ds_split['test']}
# Use 20k rows — enough for strong results, trains in ~2 hours
ds['train'] = ds['train'].select(range(18000))
ds['test']  = ds['test'].select(range(2000))
print(f"Train: {len(ds['train'])}  |  Test: {len(ds['test'])}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __len__(self):  return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Labels: 0POS=positive(0), 1NEG=negative(1)
# We convert to integers: POS→1, NEG→0 to match binary convention
def get_label(example):
    return 1 if example['label'] == '0POS' else 0

def make_dataset(split):
    data = ds[split]
    texts  = [t if isinstance(t, str) else '' for t in data['review']]
    labels = [int(l) for l in data['label']]  # 0=negative, 1=positive
    enc = tokenizer(texts, truncation=True, padding='max_length',
                    max_length=MAX_LEN, return_tensors=None)
    return SentimentDataset(enc, labels)

train_ds = make_dataset('train')
test_ds  = make_dataset('test')
print(f'Train: {len(train_ds)}  |  Test: {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

## §3 — Fine-tuning

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: 'negative', 1: 'positive'},
    label2id={'negative': 0, 'positive': 1},
).to(DEVICE)

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps)

train_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()
    for step, batch in enumerate(train_loader, 1):
        batch    = {k: v.to(DEVICE) for k, v in batch.items()}
        loss     = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total_loss += loss.item()
        if step % 100 == 0:
            print(f'  E{epoch} step {step}/{len(train_loader)}'
                  f'  loss={total_loss/step:.4f}')
    avg = total_loss / len(train_loader)
    train_losses.append(avg)
    print(f'Epoch {epoch} | avg loss {avg:.4f} | {time.time()-t0:.1f}s')

# Plot training loss
plt.figure(figsize=(6,3))
plt.plot(range(1, EPOCHS+1), train_losses, marker='o')
plt.xlabel('Epoch'); plt.ylabel('Avg training loss')
plt.title('Training loss — bert-base-chinese (Weibo)')
plt.tight_layout(); plt.savefig('training_loss.png', dpi=150); plt.show()
print('Saved training_loss.png')

## §4 — Evaluation and IPFA demonstration

In [ ]:
# Standard evaluation on held-out test set
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        preds  = model(**batch).logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch['labels'].cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f'Test accuracy: {acc:.4f}\n')
print(classification_report(all_labels, all_preds,
                             target_names=['negative', 'positive']))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['neg','pos'], yticklabels=['neg','pos'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix — test set')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# ── IPFA demonstration ────────────────────────────────────────────────────────
# These five examples illustrate the classification failure the IPFA
# research pipeline is designed to address.
#
# Examples 2, 3, and 5 use face-preserving circumlocution and affective
# attenuation — pragmatic strategies common in high-context Chinese communication
# that suppress surface distress markers while preserving underlying illocutionary
# intent.  A vanilla fine-tuned classifier is expected to misclassify these
# as neutral/positive, demonstrating the IPFA problem directly.

ipfa_examples = [
    # (text,                                          ground_truth_clinical_label, note)
    ('今天天气不错，心情还好。',                        'positive',  'Surface positive — unambiguous'),
    ('我最近有点累，可能是工作太忙了。',                 'distress',  'Attenuated distress via externalisation'),
    ('没什么，我还好，只是有点想家而已。',               'distress',  'Face-preserving minimisation ("only a little")'),
    ('这个产品非常好用，我很满意！',                    'positive',  'Explicit positive — control'),
    ('感觉最近做什么都没意思，也不知道为什么。',          'distress',  'Anhedonia expressed as vague disinterest'),
]

def predict_single(text):
    enc = tokenizer(text, return_tensors='pt',
                    truncation=True, max_length=MAX_LEN).to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    label = model.config.id2label[int(logits.argmax())]
    return label, probs

print(f'{"Text":<42} {"Model":<10} {"Clinical":<10} {"p(neg)":<8} {"Note"}')
print('-' * 95)
for text, clinical, note in ipfa_examples:
    label, probs = predict_single(text)
    flag = '⚠ IPFA miss' if clinical == 'distress' and label == 'positive' else ''
    print(f'{text:<42} {label:<10} {clinical:<10} {probs[0]:.3f}    {note}  {flag}')

print()
print('Interpretation: rows flagged ⚠ IPFA miss represent the exact failure')
print('mode the full DANN adaptation pipeline is designed to correct.')

In [ ]:
# Save metrics and model
os.makedirs('results', exist_ok=True)
report = classification_report(all_labels, all_preds,
                                target_names=['negative','positive'],
                                output_dict=True)
with open('results/metrics.json','w') as f:
    json.dump({'accuracy': round(acc,4), 'classification_report': report}, f, indent=2)
print('Saved results/metrics.json')

model.save_pretrained('model'); tokenizer.save_pretrained('model')
print('Model saved to model/')

---
## Results summary

Expected performance on ChnSentiCorp (3 epochs, bert-base-chinese):

| Metric | Expected |
|---|---|
| Accuracy | 94–96% |
| Macro F1 | 0.94–0.96 |

These figures confirm the baseline is well-calibrated on surface-level sentiment. The IPFA demonstration above shows where it breaks — motivating the DANN domain-adversarial adaptation layer in the full research pipeline.

## Next steps in the IPFA pipeline

1. Extend to multimodal input (audio + video + text) using Qwen2-Audio + OpenFace 2.0
2. Construct synthetic IPFA attenuation on CH-SIMS v2 to simulate clinical suppression
3. Apply DANN (Ganin et al., 2016) with C-drama source corpora (MAFW, MER 2024)
4. Validate on held-out CMDC + PDCH clinical datasets